# ⚙️ C-MAPSS RUL Estimation & Predictive Maintenance
## Model: Multi-Layer Perceptron (MLP Baseline)
**Assigned Team Member**: Mayurika Sathish (Member 1)  
**Course**: ICT-4442 Deep Learning Mini Project  

---
### 📌 Notebook Description
Data preprocessing, engine-wise train/val split, and MLP baseline training.


In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src.data_loader import load_raw_data, add_piecewise_rul, get_informative_features, split_train_val_by_engine, scale_features, create_sliding_windows, build_test_dataset
from src.dataset import get_dataloaders
from src.models.mlp import MLPBaseline
from src.train import calculate_metrics

In [ ]:
# 1. Load Data & Prepare Preprocessing Pipeline
train_df, test_df, rul_df = load_raw_data(dataset_id='FD001')
train_df = add_piecewise_rul(train_df, max_rul=125)
feature_cols = get_informative_features(train_df, drop_constant=True)
print(f'Selected {len(feature_cols)} informative sensor features: {feature_cols}')

In [ ]:
# 2. Engine-wise Train/Val Splitting & Feature Scaling
train_split, val_split = split_train_val_by_engine(train_df, val_ratio=0.2, seed=42)
train_scaled, val_scaled, test_scaled, scaler = scale_features(train_split, val_split, test_df, feature_cols)
X_train, y_train = create_sliding_windows(train_scaled, window_size=30, feature_cols=feature_cols)
X_val, y_val = create_sliding_windows(val_scaled, window_size=30, feature_cols=feature_cols)
X_test, y_test = build_test_dataset(test_scaled, rul_df, window_size=30, feature_cols=feature_cols, max_rul=125)
print(f'Window Data Shapes -> Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}')

In [ ]:
# 3. Train MLP Baseline Model (Member 1: Mayurika Sathish)
from src.train import train_model
results_mlp = train_model(model_type='mlp', dataset_id='FD001', epochs=25, lr=1e-3)